# RiskBands Premium Benchmark

Este notebook mostra, com dados sintéticos de crédito por safra, por que um binning que parece forte no agregado pode ficar frágil quando olhamos estabilidade temporal, cobertura, reversões e volatilidade de comportamento entre vintages.

A comparação usa três lentes:

- `OptimalBinning` puro como baseline externa
- `RiskBands` estático como baseline interna sem foco temporal
- `RiskBands` selecionado por uma régua balanceada e orientada a crédito

O objetivo aqui não é propaganda. Em alguns cenários, o candidato estático continua sendo a melhor escolha. Em outros, o RiskBands troca para uma alternativa com menos IV bruto, porém mais defensável no tempo.


In [1]:
from IPython.display import Markdown, display
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

import pandas as pd

ROOT = Path(r'D:\0_CienciaDados\1_Frameworks\RiskBands')
MODULE_PATH = ROOT / 'examples' / 'pd_vintage_benchmark' / 'pd_vintage_benchmark.py'
spec = spec_from_file_location('riskbands_benchmark_demo', MODULE_PATH)
module = module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = module
spec.loader.exec_module(module)

BENCHMARK_OBJECTIVE_KWARGS = module.BENCHMARK_OBJECTIVE_KWARGS
build_benchmark_visuals = module.build_benchmark_visuals
run_benchmark_suite = module.run_benchmark_suite

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)

suite = run_benchmark_suite(samples_per_period=180)
suite['scenario_summary']

c:\Users\JM\AppData\Local\anaconda3\envs\riskbands\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,scenario,scenario_title,best_temporal_candidate,best_balanced_candidate,selected_candidate,selected_equals_static,external_iv,static_iv,selected_iv,external_temporal_score,static_temporal_score,selected_temporal_score,external_objective_score,static_objective_score,selected_objective_score,selected_advantage_vs_static,selected_advantage_vs_external,selected_ranking_reversals,selected_basis,selected_alert_flags
0,composition_shift,Composition Shift,riskbands_temporal_quantile,riskbands_static,riskbands_static,True,1.596312,1.596312,1.596312,-1.075364,0.0,0.000000,-2.602180,0.040856,0.040856,0.000000,2.643036,0,discrimination-led,low_coverage
1,stable_credit,Stable Credit,riskbands_static_compact,riskbands_static,riskbands_static,True,1.594915,1.594915,1.594915,-0.167585,0.0,0.000000,-1.121758,0.050466,0.050466,0.000000,1.172224,0,discrimination-led,low_coverage
2,temporal_reversal,Temporal Reversal,riskbands_temporal_quantile,riskbands_temporal_quantile,riskbands_temporal_quantile,False,1.677150,1.677150,1.130749,-0.775519,0.0,0.316535,-2.084651,0.056993,0.306232,0.249239,2.390883,0,balanced,event_rate_volatility;woe_volatility;share_instability


## Como ler este benchmark

- `Stable Credit`: cenário em que a auditoria temporal não encontra motivo forte para abandonar o candidato mais discriminante.
- `Temporal Reversal`: cenário em que a leitura agregada engana e a versão balanceada do RiskBands troca para um candidato mais robusto.
- `Composition Shift`: cenário em que a mudança de composição piora a baseline externa, mas o trade-off ainda pode manter a escolha estática interna.

A régua balanceada deste benchmark usa pesos explícitos mais alinhados a crédito, dando mais importância à estabilidade temporal e mais punição para cobertura fraca, volatilidade e shortfall temporal.


## Glossário prático de leitura

### Ordem recomendada de leitura

1. **Comece pelo `scenario_summary`**: ele resume quem venceu em cada cenário.
2. **Depois olhe o `Approach board`**: aqui está a comparação direta entre baseline externa, RiskBands estático e RiskBands selecionado.
3. **Em seguida veja os gráficos por vintage**: são eles que mostram onde o agregado engana.
4. **Só depois leia o racional final**: ele explica por que a escolha vencedora foi mantida ou trocada.

### Como interpretar as principais métricas

- **IV**: força de separação agregada. É importante, mas sozinho não basta.
- **KS**: separação entre bons e maus em um corte agregado.
- **temporal_score**: score resumido de robustez temporal. Quanto maior, melhor.
- **objective_score**: score final balanceado do benchmark. Ele combina discriminação e estabilidade, com penalizações explícitas.
- **total_penalty**: soma das punições estruturais. Quanto menor, melhor.
- **coverage_ratio_min**: pior cobertura observada entre bins/vintages. Quando muito baixo, o binning pode ficar frágil em determinadas safras.
- **rare_bin_count**: número de bins raros. Muitos bins raros costumam indicar fragilidade operacional.
- **ranking_reversal_period_count**: quantas vezes a ordenação esperada entre bins se perde no tempo.
- **alert_flags**: alertas qualitativos que ajudam a entender por que um candidato parece arriscado.

### Regra de bolso para não se perder

- **IV alto + temporal ruim**: solução forte no agregado, mas potencialmente perigosa em produção.
- **IV um pouco menor + temporal claramente melhor**: muitas vezes é a solução mais defensável para crédito.
- **IV alto + temporal aceitável**: caso em que o RiskBands pode simplesmente validar a solução mais forte.


In [2]:
display(Markdown('### Objective config usado no benchmark'))
pd.Series(BENCHMARK_OBJECTIVE_KWARGS, dtype='object')

### Objective config usado no benchmark

base_weights                                                          {'separability': 0.2, 'iv': 0.2, 'ks': 0.05, 'temporal_score': 0.55}
penalty_weights    {'coverage_gap': 0.35, 'event_rate_volatility': 0.35, 'woe_volatility': 0.12, 'share_volatility': 0.25, 'ranking_rev...
minimums                                                                        {'iv': 0.02, 'temporal_score': 0.1, 'coverage_ratio': 0.8}
dtype: object

In [3]:
display(Markdown('### Configuração do objetivo usada no benchmark'))
pd.Series(BENCHMARK_OBJECTIVE_KWARGS, dtype='object')

### Configuração do objetivo usada no benchmark

base_weights                                                          {'separability': 0.2, 'iv': 0.2, 'ks': 0.05, 'temporal_score': 0.55}
penalty_weights    {'coverage_gap': 0.35, 'event_rate_volatility': 0.35, 'woe_volatility': 0.12, 'share_volatility': 0.25, 'ranking_rev...
minimums                                                                        {'iv': 0.02, 'temporal_score': 0.1, 'coverage_ratio': 0.8}
dtype: object

In [4]:
def display_scenario(result):
    spec = result['scenario_spec']
    winner = result['winner_summary'][[
        'best_static_candidate',
        'best_temporal_candidate',
        'best_balanced_candidate',
        'selected_candidate',
        'winner_margin',
        'winner_rationale',
    ]]
    board = result['approach_board'][[
        'approach_label',
        'candidate_name',
        'iv',
        'ks',
        'temporal_score',
        'objective_score',
        'total_penalty',
        'coverage_ratio_min',
        'rare_bin_count',
        'ranking_reversal_period_count',
        'alert_flags',
    ]]
    profiles = result['candidate_profiles'][[
        'candidate_name',
        'candidate_role',
        'static_profile_score',
        'temporal_profile_score',
        'balanced_profile_score',
        'static_rank',
        'temporal_rank',
        'balanced_rank',
    ]].sort_values(['balanced_rank', 'temporal_rank', 'static_rank'])

    display(Markdown(f'## {spec.title}'))
    display(Markdown(spec.description))
    display(Markdown('### Quadro comparativo das abordagens'))
    display(board)
    display(Markdown('### Vencedores por perfil'))
    display(winner)
    display(Markdown('### Resumo dos perfis candidatos'))
    display(profiles)
    display(Markdown('### Leituras-chave'))
    for takeaway in result['credit_takeaways']:
        display(Markdown(f'- {takeaway}'))

    display(Markdown(
        '### Como ler os gráficos abaixo\n'
        '- **Approach board**: resumo visual das métricas por abordagem.\n'
        '- **Trade-offs**: mostra a tensão entre discriminação e robustez temporal.\n'
        '- **Event rate por bin**: ajuda a enxergar reversões e instabilidade.\n'
        '- **Heatmap**: destaca se a estrutura por vintage continua coerente.\n'
        '- **Agregado vs vintages**: mostra quando o agregado esconde problemas.\n'
        '- **Penalizações**: explica por que um candidato perdeu.\n'
        '- **Distribuição do score**: ajuda a entender cobertura e cortes.\n'
        '- **Sampling preview**: leitura auxiliar, não central.'
    ))

    figures = build_benchmark_visuals(result)
    display(figures['benchmark_board'])
    display(figures['metric_comparison'])
    display(figures['event_rate_curves'])
    display(figures['selected_heatmap'])
    display(figures['aggregate_vs_vintage'])
    display(figures['penalty_breakdown'])
    display(figures['score_distribution'])
    display(figures['sampling_preview'])

## Leitura guiada dos resultados

### 1. `Stable Credit`
Aqui a mensagem principal é: **nem sempre o temporal precisa vencer**.

- O `RiskBands` **não trocou** o candidato final.
- A leitura temporal não encontrou evidência suficiente para abrir mão da discriminação agregada.
- Esse é um cenário importante porque mostra que a biblioteca não foi desenhada para sempre punir o candidato mais forte no agregado.

### 2. `Temporal Reversal`
Aqui está o caso mais pedagógico do notebook.

- A baseline externa e a baseline estática interna preservam **IV alto**.
- Mesmo assim, a leitura por vintage mostra piora estrutural e maior fragilidade.
- O candidato selecionado pelo RiskBands aceita perder parte do IV bruto para ganhar robustez temporal, cobertura mínima e estabilidade de ordenação.

Em outras palavras: **foi o cenário em que maximizar IV contou uma história incompleta**.

### 3. `Composition Shift`
Aqui a composição da amostra muda ao longo do tempo, mas isso **não força automaticamente** a troca de candidato.

- A baseline externa piora mais.
- O líder temporal melhora a leitura de robustez.
- Mesmo assim, o score balanceado entende que o trade-off final ainda favorece a solução estática interna.

Esse cenário é útil porque mostra que **mudança de composição não implica troca automática**.

### Como interpretar a decisão final

Quando o `selected_candidate` muda, a biblioteca está dizendo que a robustez temporal passou a ser relevante o bastante para superar a perda de IV.

Quando o `selected_candidate` não muda, a leitura correta não é “o temporal falhou”, mas sim:

> a auditoria temporal foi feita e concluiu que, naquele cenário, a solução mais discriminante ainda continua defensável.


## Limitações desta demo

- Os cenários são sintéticos e foram desenhados para expor trade-offs metodológicos, não para substituir validação em base real.
- A régua balanceada usada aqui é explícita e auditável, mas continua sendo uma escolha de benchmark; outras carteiras podem pedir pesos e pisos diferentes.
- O `TargetSampler` entra só como preview opcional para mostrar como prevalência por safra pode distorcer a leitura agregada. Ele não é o tema central do benchmark.
- A prova mais forte de valor ainda será repetir esta mesma leitura em uma variável real de bureau, scorecard ou comportamento.


## Próximo passo natural

Se este benchmark fizer sentido para a sua carteira, o passo seguinte é repetir a mesma estrutura com uma variável real de bureau ou scorecard, mantendo as mesmas tabelas de auditoria e a mesma leitura por vintage.

A leitura recomendada em base real é:

1. comparar uma solução estática pura com uma solução auditada temporalmente;
2. checar se o ganho de IV continua existindo por safra;
3. identificar bins frágeis, reversões e quedas de cobertura;
4. decidir com base em discriminação **e** robustez, não apenas em IV agregado.


## Resumo executivo do que este notebook mostrou

### Resultado mais importante
O cenário mais didático foi **`Temporal Reversal`**.

Nele, a baseline externa e a baseline estática mantêm **IV maior**, mas o candidato selecionado pelo RiskBands vence no score final porque apresenta leitura temporal mais robusta.

### Tradução prática
Isso significa que o RiskBands não está tentando “ganhar” sempre em IV. Ele está tentando responder uma pergunta mais útil para crédito:

> **qual binning continua defensável quando a carteira muda ao longo do tempo?**

### O que os três cenários sugerem em conjunto

- **`Stable Credit`**: o RiskBands pode validar a escolha estática.
- **`Temporal Reversal`**: o RiskBands pode preferir um candidato com menos IV, mas mais robusto.
- **`Composition Shift`**: a baseline externa sofre mais, mas a solução estática interna ainda pode continuar sendo a melhor escolha final.

Esse conjunto é importante porque evita uma narrativa simplista. O projeto não está dizendo que o binning temporal sempre vence; ele está mostrando **quando** a camada temporal muda a decisão e **quando** ela apenas confirma a decisão mais simples.
